# 文本分类：从 TF-IDF 到 BERT
> 难度标签：高级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：08_自然语言处理 | 源文件：文本分类.py | 核心功能：传统 ML 文本分类、特征提取 + 分类器

## 概述

文本分类是 NLP 最常见的任务。传统方法：TF-IDF + SVM/朴素贝叶斯。现代方法：BERT 微调。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
# --- 导入库 ---
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import make_pipeline

## 数学原理

### 1. 文本分类的数学定义

文本分类是将文本 $x$ 映射到预定义类别集合 $\{c_1, c_2, \ldots, c_C\}$ 中的某一个：

$$\hat{y} = \arg\max_{c_k} P(c_k | x)$$

### 2. 朴素贝叶斯分类器

朴素贝叶斯假设特征（词）之间条件独立：

$$P(c_k | x) \propto P(c_k) \prod_{i=1}^{|V|} P(w_i | c_k)^{x_i}$$

其中 $x_i$ 是词 $w_i$ 在文档 $x$ 中的 TF-IDF 值（或词频）。

**对数空间**避免下溢：

$$\log P(c_k | x) \propto \log P(c_k) + \sum_{i} x_i \log P(w_i | c_k)$$

**先验**：$P(c_k) = \frac{n_k}{n}$，$n_k$ 是类别 $c_k$ 的文档数。

**似然**（多项式朴素贝叶斯，带拉普拉斯平滑）：

$$P(w_i | c_k) = \frac{c(w_i, D_k) + \alpha}{\sum_{w \in V} c(w, D_k) + \alpha |V|}$$

其中 $\alpha = 1$ 是平滑参数，防止零概率。

### 3. 逻辑回归（多分类）

用 one-vs-rest 或 softmax 扩展到多分类：

$$P(y = c_k | \mathbf{x}) = \frac{\exp(\mathbf{w}_{c_k}^\top \mathbf{x} + b_{c_k})}{\sum_{j=1}^{C} \exp(\mathbf{w}_{c_j}^\top \mathbf{x} + b_{c_j})}$$

损失函数为交叉熵：

$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^{n}\sum_{k=1}^{C} y_{ik} \log P(y=c_k | \mathbf{x}_i)$$

### 4. 线性 SVM

SVM 最大化分类间隔：

$$\min_{\mathbf{w}} \frac{1}{2}\|\mathbf{w}\|^2 + C\sum_{i=1}^{n}\max(0, 1 - y_i(\mathbf{w}^\top \mathbf{x}_i + b))$$

$C$ 控制正则化强度。`LinearSVC` 使用 hinge loss，对高维稀疏的文本特征效果好。

### 5. TF-IDF + 分类器的组合

代码对比了多种组合：

| 组合 | 数学模型 |
|------|----------|
| TF-IDF + MultinomialNB | $P(c_k) \prod P(w_i|c_k)^{x_i}$ |
| TF-IDF + LogisticRegression | $P(c_k|x) = \text{softmax}(Wx+b)$ |
| TF-IDF + LinearSVC | $\arg\max_k w_k^T x + b_k$ |
| CountVectorizer + MultinomialNB | 朴素贝叶斯的原生输入 |

### 6. 混淆矩阵

混淆矩阵 $M \in \mathbb{R}^{C \times C}$：

$$M_{ij} = |\{x : y_{true}=c_i, y_{pred}=c_j\}|$$

对角线元素为正确分类数，非对角线为各类别间的混淆。

### 7. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `TfidfVectorizer()` | TF-IDF 特征矩阵 $\hat{X}$ |
| `MultinomialNB()` | 多项式朴素贝叶斯，$P(c_k)\prod P(w_i|c_k)^{x_i}$ |
| `LogisticRegression()` | Softmax 逻辑回归多分类 |
| `LinearSVC()` | 线性 SVM，hinge loss |
| `confusion_matrix()` | 混淆矩阵 $M$ |
| `classification_report()` | 精确率 $P$、召回率 $R$、F1 |
| `train_test_split(stratify=y)` | 分层抽样，保持类别比例 |

### 1. 构造多分类文本数据

在分类任务上训练模型并评估性能。

In [ ]:
categories = {
    "科技": [
        "人工智能技术正在改变世界",
        "新发布的手机处理器性能强大",
        "云计算服务帮助企业发展",
# --- 继续 ---
        "量子计算取得重大突破",
        "自动驾驶技术日趋成熟",
        "芯片制造工艺不断进步",
        "区块链技术应用场景广泛",
        "5G网络覆盖范围扩大",
# --- 继续 ---
    ],
    "体育": [
        "足球世界杯决赛精彩纷呈",
        "篮球运动员表现出色",
        "游泳比赛打破世界纪录",
# --- 继续 ---
        "奥运健儿为国争光",
        "网球选手晋级大满贯决赛",
        "田径比赛竞争激烈",
        "电竞比赛观众人数创新高",
        "马拉松赛事成功举办",
# --- 继续 ---
    ],
    "娱乐": [
        "电影票房突破十亿",
        "综艺节目收视率很高",
        "歌手发布新专辑",
# --- 继续 ---
        "明星参加慈善活动",
        "电视剧剧情精彩",
        "导演获得国际大奖",
        "演员演技获得好评",
        "音乐节阵容强大",
# --- 继续 ---
    ],
}

texts, labels = [], []
for i, (cat, docs) in enumerate(categories.items()):
    texts.extend(docs * 3)  # 重复增加样本
    labels.extend([i] * len(docs) * 3)

label_names = list(categories.keys())

### 2. 数据划分

运行 2. 数据划分 的代码，观察算法在该环节的行为。

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)
print(f"训练集: {len(X_train)}, 测试集: {len(X_test)}")
print(f"类别分布: {dict(zip(label_names, [labels.count(i) for i in range(len(label_names))]))}")

### 3. 不同模型对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n=== 不同模型 + 特征组合 ===")
for vec_name, vec_cls, vec_params in [
    ("CountVec", CountVectorizer, {}),
    ("TF-IDF", TfidfVectorizer, {}),
    ("TF-IDF(1,2)", TfidfVectorizer, {"ngram_range": (1, 2)}),
# --- 继续 ---
]:
    for clf_name, clf in [("NB", MultinomialNB()),
                           ("LR", LogisticRegression(max_iter=1000)),
                           ("SVM", LinearSVC(max_iter=1000))]:
        pipe = make_pipeline(vec_cls(**vec_params), clf)
# --- 继续 ---
        scores = cross_val_score(pipe, texts, labels, cv=5, scoring="accuracy")
        print(f"  {vec_name:>12} + {clf_name:>3}: {scores.mean():.4f} ± {scores.std():.4f}")

### 4. 最佳模型详细评估

在测试集上评估模型性能，关注关键指标。

In [ ]:
print("\n=== 最佳模型（TF-IDF + SVM）详细评估 ===")
best_model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2)),
    LinearSVC(max_iter=1000)
)
# --- 训练模型 ---
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print(f"测试准确率: {best_model.score(X_test, y_test):.4f}")
print(f"\n分类报告:")
print(classification_report(y_test, y_pred, target_names=label_names))

### 5. 混淆矩阵

运行 5. 混淆矩阵 的代码，观察算法在该环节的行为。

In [ ]:
print("=== 混淆矩阵 ===")
cm = confusion_matrix(y_test, y_pred)
print(f"{'':>10} {'预测科技':>8} {'预测体育':>8} {'预测娱乐':>8}")
for i, name in enumerate(label_names):
    print(f"真实{name:>4}  {cm[i]}")

### 6. 预测新文本

使用训练好的模型进行预测，观察输出结果。

In [ ]:
print("\n=== 预测新文本 ===")
new_texts = [
    "最新发布的人工智能模型性能大幅提升",
    "世界杯预选赛今晚开打",
    "新电影上映首日票房破纪录",
# --- 继续 ---
]
preds = best_model.predict(new_texts)
for text, pred in zip(new_texts, preds):
    print(f"  [{label_names[pred]}] {text}")

### 7. 超参数调优

探索关键超参数对模型行为的影响。

In [ ]:
print("\n=== 超参数调优 ===")
from sklearn.model_selection import GridSearchCV
param_grid = {
    "tfidfvectorizer__ngram_range": [(1, 1), (1, 2), (1, 3)],
    "tfidfvectorizer__max_df": [0.8, 0.9, 1.0],
# --- 继续 ---
    "linearsvc__C": [0.1, 1.0, 10.0],
}
pipe = make_pipeline(TfidfVectorizer(), LinearSVC(max_iter=1000))
gs = GridSearchCV(pipe, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
gs.fit(texts, labels)
# --- 输出结果 ---
print(f"最佳参数: {gs.best_params_}")
print(f"最佳 CV 准确率: {gs.best_score_:.4f}")

print("\n=== 文本分类要点 ===")
print("- TF-IDF + SVM/LR 是经典且强大的文本分类 baseline")
print("- N-gram 可以捕捉词组信息")
print("- 数据增强: 同义词替换、回译等可提升效果")
print("- 预训练模型(BERT)在大量数据上效果更好")
# --- 输出结果 ---
print("- 类别不平衡时考虑 class_weight='balanced'")

## 关键代码解释

```python
from sklearn.pipeline import Pipeline
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000)),
    ("clf", LogisticRegression(max_iter=1000))
])
pipe.fit(X_train_texts, y_train)
```

## 注意事项

1. 中文需要先分词或用字符级特征
2. 类别不平衡需要处理（class_weight="balanced"）
3. 长文本截断或摘要

## 总结与延伸

以上代码展示了 **文本分类** 的完整流程。

**解读要点**：
- 关注 **准确率（Accuracy）** 和 **分类报告** 中的精确率/召回率/F1
- 如果训练准确率远高于测试准确率，说明存在过拟合
- 观察 **混淆矩阵**，找出模型最容易混淆的类别

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **BERT 微调**：预训练模型 + 下游任务微调
- **零样本分类**：用大语言模型直接分类
- **多标签分类**：每个文本可以属于多个类别
